# Fine-tune Moirai-MoE-base v5 -- LoRA instead of full/freeze_ffn

v2 (full fine-tune), v3 (freeze_ffn + covariates), and v4 (freeze_ffn + 4-index training data +
macro covariates + lower LR) all showed the *same* overfitting signature: validation loss
degrading while training loss kept improving, and every fine-tuned variant underperformed
zero-shot (which itself barely beat naive persistence). freeze_ffn already cut trainable
parameters to 3% (29M/935M) and still overfit -- which points at *how* those remaining
parameters were allowed to move (fully, unconstrained), not just *how many* there were.

**v5 uses LoRA** (low-rank adaptation) on the attention projections (q_proj/k_proj/v_proj/
out_proj) instead of leaving them fully trainable:

- Every other layer (FFN/MoE experts, embeddings, output head, norms) is **fully frozen** --
  more conservative than freeze_ffn, which left attention/embeddings/head completely free.
- Attention projections get a small, rank-8 trainable delta on top of the frozen pretrained
  weight (`y = base(x) + scaling * (x @ A^T @ B^T)`), with `B` zero-initialized so training
  starts **exactly at the zero-shot solution** and can only move a small, rank-constrained
  distance away from it -- architecturally unable to overfit as violently as an unconstrained
  full-rank update can.
- This required adding real LoRA support to `uni2ts` (`LoRALinear` + `inject_lora_attention` in
  `src/uni2ts/model/moirai_moe/finetune.py`, plus a new `finetune_pattern=lora`) -- nothing like
  it existed upstream.

**Same data as v4** (S&P 500 + Nasdaq + Dow + Russell 2000, VIX/10y-yield covariates, still only
backtested on S&P 500) -- v5 isolates the fine-tuning *recipe* as the one variable that changed.

**Also added**: a fixed 50/50 blend of the zero-shot and fine-tuned forecasts in the evaluation
script, as a cheap hedge -- a convex combination of two models is never worse than the worse of
the two on the data it's scored on, and often better since their errors aren't perfectly
correlated. Not weight-optimized on the test region, to avoid leakage.

**Requires a GPU runtime**: Runtime -> Change runtime type -> T4 GPU (or better).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> select a GPU, then re-run this cell.")

## 1. Clone the repo and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/Agrim-Nuware/MOIRAI-CODE.git"
if not os.path.isdir("repo"):
    !git clone $REPO_URL repo
%cd repo

In [ ]:
!pip install -q -e '.[notebook]'
!pip install -q bitsandbytes yfinance

# uni2ts pins numpy~=1.26, which downgrades Colab's preinstalled numpy 2.x.
# Colab's preinstalled pandas is built against numpy 2.x, so once numpy is
# downgraded the two are ABI-incompatible ("numpy.dtype size changed").
# Force-reinstall a matching pandas, then restart the runtime so every
# already-imported module (numpy got pulled in transitively by torch above)
# reloads consistently. This cell intentionally crashes/restarts the kernel --
# that's expected, not an error. After it restarts, just continue running
# from the next cell (installed packages and cloned files are unaffected).
!pip install -q --force-reinstall "numpy<2" "pandas>=2.0,<2.3"
import os

os.kill(os.getpid(), 9)

**The cell above deliberately restarts the Colab runtime** (to fix a numpy/pandas
version mismatch). You'll see a "session crashed" / "automatically restarted" notice --
that's expected. Once it restarts, just continue running the cells below in order;
you do **not** need to re-run the clone or pip install cells.

In [ ]:
%cd /content/repo
import numpy as np
import pandas as pd

print("numpy:", np.__version__, "| pandas:", pd.__version__)

In [ ]:
with open(".env", "w") as f:
    f.write("CUSTOM_DATA_PATH=dataset/uni2ts_storage\n")
print(open(".env").read())

## 2. Download 4 equity indices + VIX/10y yield, engineer features, split at 2024-10-01

Same data as v4 -- reuses `prepare_v4_data.py` unchanged, since v5 only changes the
fine-tuning recipe, not the data.

In [ ]:
!python dataset/sp500/prepare_v4_data.py

In [ ]:
import json
import pandas as pd

with open("dataset/sp500/split_info_v4.json") as f:
    split_info = json.load(f)

trainval_df = pd.read_csv("dataset/sp500/sp500_v4_model_input_trainval.csv", index_col=0, parse_dates=True)
train_length = split_info["train_length"]
date_offset = trainval_df.index[train_length - 1].strftime("%Y-%m-%d")
tickers = ",".join(split_info["tickers"])
target_columns = ",".join(split_info["target_suffixes"])
covariate_columns = ",".join(split_info["covariate_suffixes"])
shared_covariate_columns = ",".join(split_info["shared_covariate_columns"])

print("tickers (training diversity):", tickers)
print("eval ticker (only one backtested):", split_info["eval_ticker"])
print("train_length:", train_length)
print("lightning val offset:", split_info["lightning_val_offset"])
print("lightning val length:", split_info["lightning_val_length"])
print("final test length (>= cutoff):", split_info["final_test_len"])
print("date_offset for CSV builder:", date_offset)

## 3. Build the uni2ts HF-format dataset (4 items, target + covariate split)

In [ ]:
!python -m uni2ts.data.builder.simple SP500V4 dataset/sp500/sp500_v4_model_input_trainval.csv \
  --dataset_type wide_multivariate_covariates \
  --tickers "{tickers}" \
  --target_columns "{target_columns}" \
  --covariate_columns "{covariate_columns}" \
  --shared_covariate_columns "{shared_covariate_columns}" \
  --date_offset "{date_offset}" --freq B

## 4. Fine-tune Moirai-MoE-base with LoRA (attention only, rank 8), GPU, fp16 + 8-bit AdamW

`model.finetune_pattern=lora` freezes the entire model, then wraps every attention
q_proj/k_proj/v_proj/out_proj layer with a rank-8 LoRA adapter (`+model.lora_rank=8`,
`+model.lora_alpha=16.0` -- the `+` is required since these are new keys not in the base config).
Trainable parameter count will be far smaller than freeze_ffn's 29M -- watch for it in the printed
model summary. No `trainer.callbacks.2.patience` override -- default `patience=3` applies.

In [ ]:
lightning_val_offset = split_info["lightning_val_offset"]
lightning_val_length = split_info["lightning_val_length"]

!python -m cli.train \
  -cp conf/finetune \
  exp_name=sp500_v5_full_finetune \
  run_name=run1 \
  tf32=false \
  model=moirai_moe_1.0_R_base \
  model.patch_size=16 \
  model.context_length=512 \
  model.prediction_length=32 \
  model.num_training_steps=3000 \
  model.num_warmup_steps=50 \
  model.finetune_pattern=lora \
  +model.lora_rank=8 \
  +model.lora_alpha=16.0 \
  model.use_8bit_adam=true \
  model.lr=3e-6 \
  data=sp500 \
  data.dataset=SP500V4 \
  data.patch_size=16 \
  data.context_length=512 \
  data.prediction_length=32 \
  data.mode=MC \
  data.train_length={train_length} \
  data.distance=16 \
  val_data=sp500 \
  val_data.dataset=SP500V4_eval \
  val_data.patch_size=16 \
  val_data.context_length=512 \
  val_data.prediction_length=32 \
  val_data.mode=MC \
  val_data.offset={lightning_val_offset} \
  val_data.eval_length={lightning_val_length} \
  val_data.distance=32 \
  trainer.max_epochs=30 \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  trainer.precision=16-mixed \
  +trainer.log_every_n_steps=10 \
  train_dataloader.batch_size=4 \
  train_dataloader.num_workers=0 \
  val_dataloader.batch_size=2 \
  val_dataloader.num_workers=0

## 5. Evaluate: zero-shot vs LoRA fine-tuned vs naive persistence vs blend, on S&P 500's held-out region

`--lora_rank`/`--lora_alpha` must match what was used for fine-tuning (defaults already match
the cell above) -- the script re-injects the same LoRA structure into a fresh pretrained module
before loading the checkpoint's weights, since the checkpoint's parameter names/shapes only
match a LoRA-wrapped module, not the plain one.

In [ ]:
!python dataset/sp500/evaluate_finetuned_v5.py \
  --context_length 512 --prediction_length 32 --num_samples 150 \
  --lora_rank 8 --lora_alpha 16.0

In [ ]:
from IPython.display import Image, display

print("Forecast comparison (Close price -- direct vs reconstructed vs naive vs blend):")
display(Image("dataset/sp500/results_v5_forecast_plot.png"))
print("\nError comparison by variate:")
display(Image("dataset/sp500/results_v5_metrics_bar.png"))
print("\nFine-tuning loss curve:")
display(Image("dataset/sp500/results_v5_loss_curve.png"))

In [ ]:
import json

with open("dataset/sp500/results_v5_metrics.json") as f:
    results = json.load(f)

naive, zs, ft, blend = results["naive"], results["zero_shot"], results["fine_tuned"], results["blend"]
print(f"{'metric':<22} {'naive':>12} {'zero-shot':>12} {'fine-tuned':>12} {'blend':>12}")
print(f"{'Close MAPE':<22} {naive['Close']['mape']:>11.2f}% {zs['Close']['mape']:>11.2f}% {ft['Close']['mape']:>11.2f}% {blend['Close']['mape']:>11.2f}%")
print(f"{'Close(recon) MAPE':<22} {naive['Close_reconstructed']['mape']:>11.2f}% {zs['Close_reconstructed']['mape']:>11.2f}% {ft['Close_reconstructed']['mape']:>11.2f}% {blend['Close_reconstructed']['mape']:>11.2f}%")
print(f"{'Return MAE':<22} {naive['Return']['mae']:>12.5f} {zs['Return']['mae']:>12.5f} {ft['Return']['mae']:>12.5f} {blend['Return']['mae']:>12.5f}")
print(f"{'LogReturn MAE':<22} {naive['LogReturn']['mae']:>12.5f} {zs['LogReturn']['mae']:>12.5f} {ft['LogReturn']['mae']:>12.5f} {blend['LogReturn']['mae']:>12.5f}")

## 6. (Optional) Save results back to your GitHub repo

Uncomment and fill in a [personal access token](https://github.com/settings/tokens) if you
want to push the plots/metrics back to your repo. Skip this if you'd rather just download
the files from the Colab file browser (left sidebar).

In [ ]:
# GITHUB_TOKEN = ""  # paste a token with repo write access, or leave blank to skip
# if GITHUB_TOKEN:
#     !git add dataset/sp500/results_v5_*.png dataset/sp500/results_v5_metrics.json
#     !git commit -m "Add Colab v5 fine-tuning results"
#     !git push https://$GITHUB_TOKEN@github.com/Agrim-Nuware/MOIRAI-CODE.git HEAD:main